# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/h0pi/flyrank-ml-intenrship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is content-performance prioritisation. The task is a ranking problem: given a set of content pages, rank them by which pages the content team should review first.

The output is a priority score, not an automatic decision to change or remove a page. The top-ranked pages will be reviewed by a human who can decide whether to refresh content, investigate a technical issue, improve internal linking, or leave the page unchanged.

Ranking fits better than simple classification because the team has limited time and needs an ordered review queue, not only a yes/no label.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The starter-data proxy target is `is_declining_proxy`.

It is created from the observed `trend_direction` field:
- `1` when `trend_direction` is `down`
- `0` when the trend is `flat` or `up`

This is a defined rule based on an observed directional trend, not a final business-outcome label. It is useful for framing and testing the workflow, but a stronger future target would measure whether a page has a meaningful decline in traffic, conversions, or another agreed content-performance outcome over a fixed future time window.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

I will use Precision@50 as the primary success metric.

Precision@50 is the share of the 50 highest-priority pages that are actually declining according to the proxy target.

A starting target of Precision@50 >= 0.70 would mean that at least 35 of the first 50 pages reviewed are genuinely in the declining group. This is useful because the content team can act on a limited weekly review queue.

I would also compare the result with a simple fixed-rule baseline, such as reviewing pages with `trend_direction == "down"` and low recent traffic, before claiming that an ML approach adds value.

In [4]:
# Precision@50 target for the later ranking evaluation
k = 50
target_precision_at_k = 0.70

print(f"Primary metric: Precision@{k}")
print(f"Starting definition of good: >= {target_precision_at_k:.0%}")
print(f"That means at least {int(k * target_precision_at_k)} of the top {k} pages are declining.")

Primary metric: Precision@50
Starting definition of good: >= 70%
That means at least 35 of the top 50 pages are declining.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row represents one anonymised content page at the measurement snapshot used in the starter data.

Each row contains page-level signals that may help prioritise review, such as traffic or engagement measures, content attributes, and a directional trend field. The data is anonymised: no client names, URLs, or private search queries are used.

The dataframe below loads the content lane slice, creates the proxy target, and displays the available columns and a small sample of rows.

In [7]:
from pathlib import Path
import pandas as pd

# Pronađi CSV fajlove samo u repozitoriju, ne u sistemskim folderima
repo_root = Path.cwd()

csv_files = [
    p for p in repo_root.rglob("*.csv")
    if "usr/local" not in str(p)
    and "site-packages" not in str(p)
    and "usr/share" not in str(p)
]

print("CSV files found in the repository:")
for path in csv_files:
    print("-", path)

# Odaberi fajl koji sadrži trend_direction
DATA_PATH = None

for path in csv_files:
    try:
        preview = pd.read_csv(path, nrows=5)
        if "trend_direction" in preview.columns:
            DATA_PATH = path
            break
    except Exception:
        pass

if DATA_PATH is None:
    raise FileNotFoundError(
        "Nisam pronašao CSV koji sadrži kolonu 'trend_direction'. "
        "Provjeri da li je starter data učitan u repo."
    )

df = pd.read_csv(DATA_PATH)

df["is_declining_proxy"] = (
    df["trend_direction"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("down")
    .astype(int)
)

print(f"\nLoaded: {DATA_PATH}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print("\nColumns:")
print(df.columns.tolist())

display(df.head(10))

print("\nProxy target distribution:")
display(
    df["is_declining_proxy"]
    .value_counts(dropna=False)
    .rename_axis("is_declining_proxy")
    .reset_index(name="page_count")
)

CSV files found in the repository:
- /content/sample_data/california_housing_test.csv
- /content/sample_data/mnist_train_small.csv
- /content/sample_data/california_housing_train.csv
- /content/sample_data/mnist_test.csv


FileNotFoundError: Nisam pronašao CSV koji sadrži kolonu 'trend_direction'. Provjeri da li je starter data učitan u repo.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule can flag pages with a downward trend, but it cannot easily decide which of many declining pages deserves attention first.

The priority depends on several signals that can interact: the direction and size of change, current traffic level, engagement or conversion signals, page age, content characteristics, and possible data noise. For example, a small decline on a high-traffic page may be more important than a large decline on a low-traffic page.

A ranking model can learn how these signals combine and produce a consistent review order. It remains decision-support, not automatic action: a human reviews the evidence before making content changes.

The model must beat a transparent fixed-rule baseline on Precision@50. If it does not, the simpler rule should be preferred.

In [6]:
# Transparent baseline: prioritise pages already marked as trending down.
# This is a baseline queue, not a trained model.

baseline_queue = (
    df.assign(
        baseline_priority=df["is_declining_proxy"]
    )
    .sort_values("baseline_priority", ascending=False)
    .head(50)
)

baseline_precision_at_50 = baseline_queue["is_declining_proxy"].mean()

print("Baseline rule: put pages with trend_direction == 'down' first.")
print(f"Baseline Precision@50: {baseline_precision_at_50:.1%}")
print("A future ranking model should be compared with this baseline before it is used.")

KeyError: 'is_declining_proxy'

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.